In [0]:
library(dataiku)
library(stats) # need this to calculate Mahalanobis Distance

In [0]:
# Recipe inputs
counterfactual_test_data <- dkuReadDataset("counterfactual_test_data", samplingMethod="head", nbRows=100000)
hurdle_predictions_testing <- dkuManagedFolderPath("5NPBmWH1")

In [0]:
colnames(counterfactual_test_data)

In [0]:
# variables I'm interested in for matching:
match_vars  <- c('blue_ss_frac',
                    'blue_ls_frac',
                    'red_ls_frac',
                    'orange_ls_frac',
                    'yellow_ss_frac',
                    'red_ss_frac',
                    'orange_ss_frac',
                    'yellow_ls_frac',
                    'roof_strong_wall_strong',
                    'roof_strong_wall_light',
                    'roof_strong_wall_salv',
                    'roof_light_wall_strong',
                    'roof_light_wall_light',
                    'roof_light_wall_salv',
                    'roof_salv_wall_strong',
                    'roof_salv_wall_light',
                    'roof_salv_wall_salv'
                   )

# Convert to matrix (important for Mahalanobis computation)
match_matrix <- as.matrix(counterfactual_test_data[, match_vars])

# Ensure there are no NA values (Mahalanobis fails with NAs)
match_matrix <- na.omit(match_matrix)

# Compute covariance matrix of the data
cov_matrix <- cov(match_matrix)

# Compute pairwise Mahalanobis distance manually
mah_dist <- as.matrix(proxy::dist(match_matrix, method = "mahalanobis", cov = cov_matrix))

# Get the nearest neighbor for each row (excluding itself)
nearest_match_index <- apply(mah_dist, 1, function(x) which.min(ifelse(x == 0, Inf, x)))

# Create a matched dataset
matched_df <- counterfactual_test_data[nearest_match_index, ]

In [0]:
head(matched_df)

In [0]:
# Recipe outputs
matching_counterfactuals <- dkuManagedFolderPath("ZO3oPxC1")